In [1]:
# Clonar tu repositorio (cambia la URL por la tuya)
!git clone -b for-review https://github.com/Andmo2004/DBSCAN-TO-MIL-TFG

# Movernos a la carpeta del proyecto (Kaggle usa %cd en lugar de cd)
%cd DBSCAN-TO-MIL-TFG

Cloning into 'DBSCAN-TO-MIL-TFG'...
remote: Enumerating objects: 487, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 487 (delta 27), reused 165 (delta 19), pack-reused 292 (from 1)
Receiving objects: 100% (487/487), 39.90 MiB | 23.53 MiB/s, done.
Resolving deltas: 100% (153/153), done.
/kaggle/working/DBSCAN-TO-MIL-TFG


In [2]:
# Instalar uv
!pip install uv

# Usar uv para instalar todas tus dependencias al instante
!uv pip install --system -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 58.5 MB/s eta 0:00:00
Using Python 3.12.12 environment at: /usr
Checked 8 packages in 253ms


In [3]:
# Asegurarnos de que las carpetas existen
import os
os.makedirs("results", exist_ok=True)
os.makedirs("datasets", exist_ok=True)

# Comprobar que los ARFF están ahí
!ls datasets/

BirdsChestnut-backedChickadee.arff  musk2.arff
BirdsHammondsFlycatcher.arff	    mutagenesis3_atoms.arff
dummys_datasets_gen.py		    mutagenesis3_chains.arff
Harddrive1.arff			    Newsgroups1.arff
ImageElephant.arff		    simple_dummy.arff
musk1.arff			    Thioredoxin.arff


# PRECALCULO DE LAS MATRICES DE DISTANCIA
#### (if Not Exists)

In [4]:
!python experiments/00_distance_matrix_cache.py

PRECOMPUTANDO MATRICES DE DISTANCIA (CACHÉ PERSISTENTE)

► Procesando Dataset: musk1
NumExpr defaulting to 4 threads.
Cargando dataset desde ARFF: datasets/musk1.arff
Iniciando carga de 'datasets/musk1.arff' como dataset 'musk1'
Extrayendo esquema de instancias desde 'datasets/musk1.arff'
Esquema extraído: 166 atributos
Construyendo bolsas...
Bolsas construidas: 92 bolsas, 476 instancias totales
Carga completada: 92 bolsas procesadas
Entrenando MinMaxScaler en dataset 'musk1_train'
MinMaxScaler entrenado: min=[  -9. -199. -166.]..., max=[82. 98. 83.]...
Transformando dataset 'musk1_train' con MinMaxScaler
Entrenando StandardScaler en dataset 'musk1_train'
StandardScaler entrenado: mean=[  39.81458967 -118.89057751  -73.74772036]..., std=[16.46324911 87.46629504 69.20626983]...
Transformando dataset 'musk1_train' con StandardScaler

► Procesando Dataset: musk2
Cargando dataset desde ARFF: datasets/musk2.arff
Iniciando carga de 'datasets/musk2.arff' como dataset 'musk2'
Extrayendo esquem

---
## Fase 1 — Caracterización del Problema (EDA)

**Propósito:** Justificar la dificultad intrínseca de los datasets seleccionados antes de ejecutar ningún modelo. El TFG debe demostrar que el problema no es trivial.

In [5]:
!python experiments/01_data_analisis_phase.py


  INICIANDO FASE 1: EDA Y CARACTERIZACIÓN DEL PROBLEMA

► Analizando Dataset: musk1...
  - Obteniendo matriz Hausdorff de la caché...
  - Obteniendo matriz Cauchy-Schwarz de la caché...
  > Bags: 92 (Pos: 47, Neg: 45)
  > Instancias (Min/Avg/Max): 2 / 5.2 / 40
  > Sep Ratio (Hausdorff): 1.0408 | (Cauchy-Schwarz): 1.1038
  [!] ADVERTENCIA: Solapamiento severo detectado (Sep Ratio < 1.2). Dataset de alta dificultad.

► Analizando Dataset: musk2...
  - Obteniendo matriz Hausdorff de la caché...
  - Obteniendo matriz Cauchy-Schwarz de la caché...
  > Bags: 102 (Pos: 39, Neg: 63)
  > Instancias (Min/Avg/Max): 1 / 64.7 / 1044
  > Sep Ratio (Hausdorff): 0.9733 | (Cauchy-Schwarz): 0.9279
  [!] ADVERTENCIA: Solapamiento severo detectado (Sep Ratio < 1.2). Dataset de alta dificultad.

► Analizando Dataset: ImageElephant...
WARNING - 125 atributos son constantes (rango = 0)
  - Obteniendo matriz Hausdorff de la caché...
  - Obteniendo matriz Cauchy-Schwarz de la caché...
  > Bags: 200 (Pos: 100,

---
## Fase 2 — Optimización de Hiperparámetros (Tuning con Optuna)

**Propósito:** Encontrar la configuración óptima `(scaler, metric, min_pts, eps)` para cada dataset de forma sistemática y reproducible.


In [6]:
!python experiments/02_optimization_phase.py

INICIANDO FASE 2: OPTIMIZACIÓN DE HIPERPARÁMETROS

  INICIANDO BÚSQUEDA OPTUNA (100 trials por dataset)

► Procesando Dataset: musk1...
  [+] Inyectando solución previa (Warm Start): eps_percentile=9.13%
Best trial: 79. Best value: 0.785714: 100%|███| 100/100 [00:01<00:00, 85.28it/s]
  >> Mejor F1 Score : 0.7857
  >> Scaler         : MinMaxScaler
  >> Distancia      : earth_movers
  >> eps_percentile : 9.19% -> eps_abs: 2.643122
  >> min_pts        : 4
  >> Clusters Hall. : 2
  >> Ruido (%)      : 14.1%

► Procesando Dataset: musk2...
  [+] Inyectando solución previa (Warm Start): eps_percentile=4.55%
Best trial: 0. Best value: 0.71913: 100%|█████| 100/100 [00:03<00:00, 30.10it/s]
  >> Mejor F1 Score : 0.7191
  >> Scaler         : StandardScaler
  >> Distancia      : hausdorff
  >> eps_percentile : 4.55% -> eps_abs: 10.497524
  >> min_pts        : 2
  >> Clusters Hall. : 11
  >> Ruido (%)      : 23.9%

► Procesando Dataset: ImageElephant...
WARNING - 125 atributos son constantes (rango

---

## Fase 3 — Evaluación de la Calidad del Clustering (CVIs Internos)

**Propósito:** Evaluar si los clústeres detectados por MIDBSCAN son geométricamente sólidos, independientemente de las etiquetas reales. Esto es fundamental para validar DBSCAN como algoritmo no supervisado.

In [7]:
!python experiments/03_cluster_evaluation_phase.py

INICIANDO FASE 3: EVALUACIÓN DE CALIDAD DE CLUSTERING (CVIs INTERNOS)

Procesando dataset: musk1
WARNING - [VRC] Se necesitan al menos 2 clusters.
WARNING - [SED] No hay clusters reales.
WARNING - [DD] No hay clusters reales.
WARNING - [Hc] No hay clusters reales.
WARNING - [VRC] Se necesitan al menos 2 clusters.
WARNING - [IIndex] No hay clusters reales.
  [+] Evaluaciones completadas.

Procesando dataset: musk2
WARNING - [VRC] Se necesitan al menos 2 clusters.
WARNING - [SED] No hay clusters reales.
WARNING - [DD] No hay clusters reales.
WARNING - [Hc] No hay clusters reales.
WARNING - [VRC] Se necesitan al menos 2 clusters.
WARNING - [IIndex] No hay clusters reales.
  [+] Evaluaciones completadas.

Procesando dataset: ImageElephant
WARNING - 125 atributos son constantes (rango = 0)
WARNING - [VRC] Se necesitan al menos 2 clusters.
  [+] Evaluaciones completadas.

Procesando dataset: BirdsChestnut
WARNING - [VRC] Se necesitan al menos 2 clusters.
WARNING - [SED] No hay clusters reale

---

## Fase 4 — Clasificación Final y Comparativa (Baseline con MIKnn)

**Propósito:** Responder la pregunta central del TFG: ¿es MIDBSCAN competitivo frente a un clasificador supervisado de distancias en problemas MIL?

In [8]:
!python experiments/04_comparison_phase.py

INICIANDO FASE 4: CLASIFICACIÓN FINAL Y COMPARATIVA (BASELINE MIKnn)

Procesando dataset: musk1
  [+] DBSCAN F1: 0.7879 | KNN F1 (k=1): 0.8485 | Δ F1: -0.0606
  [+] Matrices de confusión generadas en /kaggle/working/DBSCAN-TO-MIL-TFG/results/confusion_matrices

Procesando dataset: musk2
  [+] DBSCAN F1: 0.6400 | KNN F1 (k=1): 0.8276 | Δ F1: -0.1876

Procesando dataset: ImageElephant
WARNING - 125 atributos son constantes (rango = 0)
  [+] DBSCAN F1: 0.7294 | KNN F1 (k=3): 0.7733 | Δ F1: -0.0439

Procesando dataset: BirdsChestnut
  [+] DBSCAN F1: 0.6780 | KNN F1 (k=3): 0.6857 | Δ F1: -0.0077

Procesando dataset: BirdsHammonds
  [+] DBSCAN F1: 0.9841 | KNN F1 (k=1): 0.9841 | Δ F1: +0.0000
  [+] Matrices de confusión generadas en /kaggle/working/DBSCAN-TO-MIL-TFG/results/confusion_matrices

Procesando dataset: Harddrive1
WARNING - 11 atributos son constantes (rango = 0)
  [+] DBSCAN F1: 0.9833 | KNN F1 (k=3): 0.9573 | Δ F1: +0.0261

Procesando dataset: mutagenesis_atoms
  [+] DBSCAN F1: 0

___

## Fase 5 — Validación Estadística

**Propósito:** Proporcionar rigor científico a las comparaciones del TFG. Un F1 mayor en la tabla no es evidencia suficiente sin una prueba de significancia estadística.


In [9]:
!python experiments/05_statistical_eval_phase.py

INICIANDO FASE 5: VALIDACIÓN ESTADÍSTICA (TEST DE WILCOXON)
[*] Cargando resultados de: full_eval_060520261424.csv

Resultados del Test de Wilcoxon de Rangos con Signo:

Comparación                              | W      | p-valor  | Sig(0.05)  | Effect r   | Interpretación
---------------------------------------------------------------------------------------------------------
DBSCAN vs KNN (F1)                       | 14.0   | 0.3594   | No         | 0.2898     | pequeño
DBSCAN vs KNN (Accuracy)                 | 14.0   | 0.6406   | No         | 0.1476     | pequeño
DBSCAN vs KNN (Recall)                   | 16.0   | 0.4961   | No         | 0.2152     | pequeño
DBSCAN vs KNN (F1 - Subgrupo A)          | 1.0    | 0.5000   | No         | 0.3894     | moderado
DBSCAN vs KNN (F1 - Subgrupo B)          | 4.0    | 0.8750   | No         | 0.0704     | pequeño

[+] Tabla de p-valores guardada en: /kaggle/working/DBSCAN-TO-MIL-TFG/results/statistical_tests/wilcoxon_results_060520261424.csv

--